# NIFTY decline and recovery: event study
The primary rule was fixed before viewing out-of-sample outcomes. This notebook reads the committed source snapshot and regenerates all outputs.

## 1. Research question and hypothesis
Does a NIFTY close-to-close fall of at least 2% predict a positive next-open to fifth-session-close return, and a return above the same-period unconditional baseline? Positive recovery and excess performance are separate claims.

## 2. Configuration
The threshold, holding period, per-side costs and slippage, overlap rule, and chronological split are in one immutable configuration object.

In [1]:
from src.research import Config, load_raw, validate, opportunities, periods, select_events, analyze, robustness, run
from IPython.display import display
import pandas as pd
cfg=Config()
print(cfg)

Config(threshold=-0.02, holding_period=5, cost_bps_per_side=5.0, slippage_bps_per_side=5.0, overlap='nonoverlap', split_date='2022-01-01', bootstrap_draws=3000, seed=20260922)


## 3. Data loading
The source is the committed Yahoo Finance ^NSEI daily chart JSON. The response is pinned with a SHA-256 hash in run metadata.

In [2]:
raw=load_raw()
print(raw.shape, raw.Date.min(), raw.Date.max())
display(raw.head())

(3337, 5) 2013-01-02 00:00:00 2026-07-07 00:00:00


,Date,Open,High,Low,Close
0,2013-01-02,5982.600098,6006.049805,5982.000000,5993.250000
1,2013-01-03,6015.799805,6017.000000,5986.549805,6009.500000
2,2013-01-04,6011.950195,6020.750000,5981.549805,6016.149902
3,2013-01-07,6042.149902,6042.149902,5977.149902,5988.399902
4,2013-01-08,5983.450195,6007.049805,5964.399902,6001.700195


## 4. Validation and cleaning
Every missing or impossible OHLC row is recorded. Suspicious moves are flagged and retained. Candidate event windows crossing a missing source bar are excluded.

In [3]:
clean,checks,flags,gaps=validate(raw)
display(checks)
display(flags)
display(gaps)

,Check,Value
0,raw_rows,3337
1,date_min,2013-01-02
2,date_max,2026-07-07
3,wrong_order_transitions,0
4,invalid_dates,0
5,duplicate_date_rows,0
6,exact_duplicate_rows,0
7,missing_ohlc_rows,18
8,invalid_ohlc_rows,0
9,excluded_rows,18


,Date,Open,High,Low,Close,Reason,CloseReturn
0,2014-01-01,NaN,NaN,NaN,NaN,missing OHLC,NaN
1,2014-02-17,NaN,NaN,NaN,NaN,missing OHLC,NaN
2,2014-03-22,NaN,NaN,NaN,NaN,missing OHLC,NaN
3,2014-04-24,NaN,NaN,NaN,NaN,missing OHLC,NaN
4,2014-10-15,NaN,NaN,NaN,NaN,missing OHLC,NaN
5,2015-01-01,NaN,NaN,NaN,NaN,missing OHLC,NaN
6,2015-02-28,NaN,NaN,NaN,NaN,missing OHLC,NaN
7,2015-04-15,NaN,NaN,NaN,NaN,missing OHLC,NaN
8,2016-01-01,NaN,NaN,NaN,NaN,missing OHLC,NaN
9,2016-08-12,NaN,NaN,NaN,NaN,missing OHLC,NaN


,Date,PreviousDate,CalendarDays,MissingOHLCSourceRows,Classification
431,2014-10-07,2014-10-01,6.0,0,calendar gap: holiday status unverified
442,2014-10-27,2014-10-22,5.0,0,calendar gap: holiday status unverified
548,2015-04-06,2015-04-01,5.0,0,calendar gap: holiday status unverified
787,2016-03-28,2016-03-23,5.0,0,calendar gap: holiday status unverified
800,2016-04-18,2016-04-13,5.0,0,calendar gap: holiday status unverified
882,2016-08-16,2016-08-11,5.0,1,data-quality: source row lacks OHLC
1282,2018-04-02,2018-03-28,5.0,0,calendar gap: holiday status unverified
2277,2022-04-18,2022-04-13,5.0,0,calendar gap: holiday status unverified


## 5. Event detection
The signal is known only after the event day's close. Entry is at the next observed session's open; exit is at the fifth session's close. A new selected signal must occur after the prior exit. Gross and net returns retain their separate meanings.

In [4]:
opp=opportunities(clean,cfg)
parts=periods(opp,cfg)
events={k:select_events(v,cfg)[0] for k,v in parts.items()}
for k,v in events.items(): print(k, 'raw / selected', select_events(parts[k],cfg)[1],len(v))
display(events['development'].head())

development raw / selected 62 44
oos raw / selected 19 14


,EventIndex,EventDate,EventReturn,EntryDate,EntryPrice,ExitDate,ExitPrice,HoldingPeriod,ForwardReturn,NetReturn
0,96,2013-05-23,-0.020912,2013-05-24,6010.700195,2013-05-30,6124.049805,5,0.018858,0.016822
1,102,2013-05-31,-0.022550,2013-06-03,5997.350098,2013-06-07,5881.000000,5,-0.019400,-0.021359
2,116,2013-06-20,-0.028571,2013-06-21,5639.899902,2013-06-27,5682.350098,5,0.007527,0.005514
3,149,2013-08-06,-0.025179,2013-08-07,5549.299805,2013-08-14,5742.299805,5,0.034779,0.032712
4,155,2013-08-16,-0.040829,2013-08-19,5497.549805,2013-08-23,5471.750000,5,-0.004693,-0.006682


## 6. Event analysis and statistical evidence
The median and tails show asymmetry obscured by the mean. Seeded event bootstrap intervals assess event means; a 20-candidate-day block bootstrap is used for the baseline mean. These intervals remain approximate because regimes and clustered crises can create further dependence.

In [5]:
summary=pd.DataFrame([r for k,v in parts.items() for r in analyze(v,cfg,k)])
display(summary[['period','return_kind','event_n','event_mean','event_median','event_std','event_win_rate','ci_low','ci_high','p_positive']])

,period,return_kind,event_n,event_mean,event_median,event_std,event_win_rate,ci_low,ci_high,p_positive
0,development,gross,44,-0.002480,0.006224,0.039287,0.522727,-0.014134,0.008280,0.668444
1,development,net,44,-0.004473,0.004214,0.039208,0.522727,-0.016103,0.006266,0.785072
2,oos,gross,14,0.008289,-0.001473,0.031339,0.428571,-0.006919,0.025024,0.150950
3,oos,net,14,0.006275,-0.003468,0.031277,0.428571,-0.008903,0.022976,0.216594


## 7. Baseline
For each chronological partition, every eligible signal date contributes the same next-open to Hth-close return. The baseline includes event dates and so represents unconditional NIFTY opportunities, not a separate regime.

In [6]:
display(summary[['period','return_kind','baseline_n','baseline_mean','baseline_median','baseline_std','baseline_win_rate','mean_difference','excess_ci_low','excess_ci_high','p_excess']])

,period,return_kind,baseline_n,baseline_mean,baseline_median,baseline_std,baseline_win_rate,mean_difference,excess_ci_low,excess_ci_high,p_excess
0,development,gross,2117,0.001459,0.002456,0.023378,0.555503,-0.003939,-0.015968,0.006862,0.748750
1,development,net,2117,-0.000542,0.000453,0.023332,0.509211,-0.003931,-0.015936,0.006848,0.748750
2,oos,gross,1083,0.001122,0.001505,0.018643,0.529086,0.007167,-0.008592,0.024006,0.187604
3,oos,net,1083,-0.000878,-0.000496,0.018606,0.493075,0.007153,-0.008575,0.023958,0.187604


## 8. Robustness
Thresholds and holding periods form a predefined grid; overlap and trading drag are varied separately. Rows are diagnostic, not choices from which to pick a winning rule.

In [7]:
rob=robustness(clean,cfg)
display(rob.loc[rob.return_kind=='net',['threshold','holding_period','overlap','cost_bps_side','slippage_bps_side','event_n','event_mean','mean_difference','excess_ci_low','excess_ci_high']])

,threshold,holding_period,overlap,cost_bps_side,slippage_bps_side,event_n,event_mean,mean_difference,excess_ci_low,excess_ci_high
1,-0.015,1,nonoverlap,5.0,5.0,110,-0.002324,0.000455,-0.002073,0.003253
3,-0.015,3,nonoverlap,5.0,5.0,97,0.001458,0.003105,-0.002540,0.008380
5,-0.015,5,nonoverlap,5.0,5.0,86,0.001929,0.002471,-0.005082,0.009324
7,-0.015,10,nonoverlap,5.0,5.0,66,0.010432,0.008495,-0.001870,0.018730
9,-0.020,1,nonoverlap,5.0,5.0,56,-0.002268,0.000510,-0.004039,0.005629
11,-0.020,3,nonoverlap,5.0,5.0,50,-0.001599,0.000048,-0.009848,0.009444
13,-0.020,5,nonoverlap,5.0,5.0,44,-0.004473,-0.003931,-0.015936,0.006848
15,-0.020,10,nonoverlap,5.0,5.0,38,0.011487,0.009550,-0.005290,0.024385
17,-0.025,1,nonoverlap,5.0,5.0,28,-0.000686,0.002092,-0.005402,0.010676
19,-0.025,3,nonoverlap,5.0,5.0,24,0.001820,0.003467,-0.013365,0.019959


## 9. Out-of-sample
The split is 2022-01-01. Development opportunities must exit before the split, while OOS signals must begin on or after it. No OOS outcome is used in the primary configuration.

In [8]:
display(summary.loc[summary.period=='oos'])

,period,return_kind,threshold,holding_period,overlap,cost_bps_side,slippage_bps_side,raw_events,eligible_days,event_n,...,baseline_max,mean_difference,median_difference,win_rate_difference,ci_low,ci_high,p_positive,excess_ci_low,excess_ci_high,p_excess
2,oos,gross,-0.02,5,nonoverlap,5.0,5.0,19,1083,14,...,0.072163,0.007167,-0.002977,-0.100514,-0.006919,0.025024,0.150950,-0.008592,0.024006,0.187604
3,oos,net,-0.02,5,nonoverlap,5.0,5.0,19,1083,14,...,0.070021,0.007153,-0.002972,-0.064503,-0.008903,0.022976,0.216594,-0.008575,0.023958,0.187604


## 10. Falsification
Check whether development excess is positive after costs, whether the lower confidence bound clears zero, whether overlap changes the result, and whether OOS persists. Missing-bar crossings and large crisis observations are disclosed. Many grid cells invite chance findings.

In [9]:
regime=pd.concat([v.assign(Period=k) for k,v in events.items()],ignore_index=True)
regime["Regime"]=pd.cut(regime.EventDate.dt.year,bins=[2012,2019,2021,2026],labels=["2013-2019","2020-2021","2022-2026"])
display(regime.groupby(["Regime","Period"],observed=True).NetReturn.agg(n="size",mean="mean",median="median"))
dev_net=summary[(summary.period=='development')&(summary.return_kind=='net')].iloc[0]
oos_net=summary[(summary.period=='oos')&(summary.return_kind=='net')].iloc[0]
print('Development net excess:',dev_net.mean_difference,'CI:',dev_net.excess_ci_low,dev_net.excess_ci_high)
print('OOS net excess:',oos_net.mean_difference,'CI:',oos_net.excess_ci_low,oos_net.excess_ci_high)

,,n,mean,median
Regime,Period,,,
2013-2019,development,27,0.000352,0.005514
2020-2021,development,17,-0.012136,-0.003396
2022-2026,oos,14,0.006275,-0.003468


Development net excess: -0.00393141823787968 CI: -0.015936145515175977 0.0068480582056948715
OOS net excess: 0.007152905452245799 CI: -0.008575209083608686 0.023958164322324203


## 11. Conditional backtest and conclusion
A simple event-driven backtest is run only if both partitions have a positive lower 95% confidence bound for net excess. A weak or negative result is an investigation finding, not a strategy to optimize.

In [10]:
result=run(cfg)
print('Decision and figures: results/tables/backtest_decision.json; results/figures/')

     period return_kind  threshold  holding_period    overlap  cost_bps_side  slippage_bps_side  raw_events  eligible_days  event_n  event_mean  event_median  event_std  event_win_rate  event_min  event_p05  event_p25  event_p75  event_p95  event_max  baseline_n  baseline_mean  baseline_median  baseline_std  baseline_win_rate  baseline_min  baseline_p05  baseline_p25  baseline_p75  baseline_p95  baseline_max  mean_difference  median_difference  win_rate_difference    ci_low  ci_high  p_positive  excess_ci_low  excess_ci_high  p_excess
development       gross  -0.020000               5 nonoverlap       5.000000           5.000000          62           2117       44   -0.002480      0.006224   0.039287        0.522727  -0.143795  -0.059738  -0.016532   0.021172   0.038741   0.057276        2117       0.001459         0.002456      0.023378           0.555503     -0.180407     -0.034102     -0.010947      0.015044      0.034825      0.111517        -0.003939           0.003768            